
# Birthplace → Education Mobility of Chinese Students

This notebook links student origins from `birth_residence.csv` to educational trajectories from `educ_geoloc.csv`, using the supplied historical province boundaries and birthplace city coordinates.

It produces:

1. **Birthplace → first education Sankey**
2. **Birthplace → first education spatial flow map**
3. **Multi-stage education Sankey**
4. **Multi-stage spatial trajectory map**, with selectable stages and optional stage colors
5. **Interactive choropleth of births by historical province**
6. **Interactive proportional-circle map of births by city/town**
7. **Interactive U.S. state choropleth of educational activity**

### Global birth-year filtering

Every interactive visualization includes a **birth-year range** filter. Because many directory entries do not report a birth year, each visualization also includes an explicit option to **include or exclude unknown birth years**.

### Geography

- Birth cities/towns use `chinese_cities_with_coordinates.csv`.
- Province boundaries use the supplied `Provinces_1912-1931` shapefile.
- The notebook uses `pyshp` + `pyproj` instead of GeoPandas, avoiding a GeoPandas dependency.
- Education locations use the coordinates already present in `educ_geoloc.csv`.


In [1]:

from pathlib import Path
from collections import Counter, defaultdict
import itertools
import json
import html
import re
import math
import shutil

import numpy as np
import pandas as pd

from IPython.display import display, IFrame

# Lightweight shapefile dependencies.
# If your local environment does not have them, run once:
# %pip install pyshp pyproj
import shapefile
from pyproj import CRS, Transformer

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

BIRTH_PATH = Path("/mnt/data/birth_residence.csv")
EDUC_PATH = Path("/mnt/data/educ_geoloc.csv")
CITY_COORD_PATH = Path("/mnt/data/chinese_cities_with_coordinates.csv")

HISTORICAL_SHP_PATH = Path("/mnt/data/Provinces_1912-1931.shp")
HISTORICAL_SHP_ALT_PATH = Path("/mnt/data/Provinces_1912-1931(1).shp")
HISTORICAL_BASE = Path("/mnt/data/Provinces_1912-1931")

OUTPUT_DIR = Path("/mnt/data/birth_education_outputs_enhanced")
OUTPUT_DIR.mkdir(exist_ok=True)

birth_raw = pd.read_csv(BIRTH_PATH, sep=";", dtype=str)
educ_raw = pd.read_csv(EDUC_PATH)

print("Birth/residence:", birth_raw.shape)
print("Education:", educ_raw.shape)
print(
    "Student overlap:",
    len(
        set(birth_raw["NameID"].dropna()) &
        set(educ_raw["NameID"].dropna())
    )
)

display(birth_raw.head())
display(educ_raw.head())


Birth/residence: (1613, 26)
Education: (3151, 21)
Student overlap: 1594


,NameID,FullName,FullName_PY,Gender,Name_WG,Alter_name,Alter_name_PY,birth_year,age_in_1944,Birth_province_zh_st,Birth_province_py_st,Birth_province_zh,Birth_province_py,Birth_town_zh,Birth_town_py,Address,Institution,Sublevel,Instit_class,Category,City,StateProv,Country,Page_src,Page_PDF,Source
0,N00001,丁康金梅,Ding Kangjinmei,F,"Ting, Ging-mei Kang",NaN,NaN,1918,26,江蘇,Jiangsu,江蘇,Jiangsu,川沙,Chuansha,"1442 Washington Heights, Ann Arbor, Michigan",NaN,NaN,NaN,NaN,Ann Arbor,Michigan,USA,1,439,近代同學錄-03_extract_旅美中國同人録_1944
1,N00002,丁文華,Ding Wenhua,F,"Ting, Mary Jean",NaN,NaN,NaN,NaN,江蘇,Jiangsu,江蘇,Jiangsu,武進,Wujin,"c/o Mrs. P.C. Shute, Accord P.O. Mass.",NaN,NaN,NaN,NaN,Accord,Massachusetts,USA,1,439,近代同學錄-03_extract_旅美中國同人録_1944
2,N00003,丁忱,Ding Chen,M,"Ting, Chen",雋詞,Juanci,1919,25,江蘇,Jiangsu,江蘇,Jiangsu,無錫,Wuxi,"0-35 Lowell House, Cambridge, Mass.",NaN,NaN,NaN,NaN,Cambridge,Massachusetts,USA,1,439,近代同學錄-03_extract_旅美中國同人録_1944
3,N00004,丁美曾,Ding Meiceng,F,"Ding, Mai-tcheng",NaN,NaN,NaN,NaN,江蘇,Jiangsu,江蘇,Jiangsu,NaN,NaN,"1078 W.35th St., los Angeles calif.",NaN,NaN,NaN,NaN,Los Angeles,California,USA,1,439,近代同學錄-03_extract_旅美中國同人録_1944
4,N00005,丁恩沐,Ding Enmu,M,"Ting, Robert",NaN,NaN,NaN,NaN,江蘇,Jiangsu,江蘇,Jiangsu,上海,Shanghai,"420 Salvatierra St., Stanford University, Calif.",Stanford University,NaN,University,Academic,Stanford,California,USA,1,439,近代同學錄-03_extract_旅美中國同人録_1944


,Unnamed: 0,NameID,FullName,FullName_PY,Name_WG,Gender,University,Department,Degree,Expanded,Level,discipline,Year,Page,University_zh,University_clean,City,Province_State,Country,Latitude,Longitude
0,1,N00001,丁康金梅,Ding Kangjinmei,"Ting, Ging-mei Kang",F,University of Michigan,NaN,NaN,NaN,NaN,Nursing,1943.0,439,NaN,University of Michigan,Ann Arbor,Michigan,United States,42.27590,-83.73100
1,2,N00002,丁文華,Ding Wenhua,"Ting, Mary Jean",F,Mount Holyoke College,NaN,NaN,NaN,NaN,Chemistry,1943.0,439,NaN,Mount Holyoke College,South Hadley,Massachusetts,United States,42.25842,-72.57453
2,3,N00004,丁美曾,Ding Meiceng,"Ding, Mai-tcheng",F,University of California,NaN,NaN,NaN,NaN,Accounting,1943.0,439,NaN,University of California,Berkeley,California,United States,37.87220,-122.27600
3,4,N00007,丁毓明,Ding Yuming,"Ting, Yoeh-ming",F,Mount Holyoke College,NaN,NaN,NaN,NaN,Medicine,1943.0,439,NaN,Mount Holyoke College,South Hadley,Massachusetts,United States,42.25842,-72.57453
4,5,N00020,王式好,Wang Shihao,"Hao, JoyceWang",F,State University of Iowa,NaN,NaN,NaN,NaN,Chemistry,1943.0,440,NaN,State University of Iowa,Iowa City,Iowa,United States,41.66270,-91.55490


## 1. Clean and harmonize data

In [2]:

def clean_string(v):
    return "" if pd.isna(v) else str(v).strip()

def normalize_country(country):
    c = clean_string(country).lower()

    if c in {
        "china", "people's republic of china",
        "pr china", "p.r. china", "republic of china"
    }:
        return "China"

    if c in {
        "united states", "united states of america",
        "usa", "u.s.", "us", "u.s.a."
    }:
        return "US"

    if not c:
        return "Unknown"

    return "Other"

def normalize_level(v):
    if pd.isna(v) or not str(v).strip():
        return "Unspecified"

    level = str(v).strip()

    if level.lower() in {"doctor", "doctorate", "licentiate"}:
        return "Doctorate"

    if level.lower() in {"diploma", "certificate"}:
        return "Other degree"

    return level

birth = birth_raw.copy()
birth["_student"] = birth["NameID"].astype(str).str.strip()
birth["_birth_year"] = pd.to_numeric(birth["birth_year"], errors="coerce")
birth["_birth_province"] = birth["Birth_province_py"].fillna("").astype(str).str.strip()
birth["_birth_city"] = birth["Birth_town_py"].fillna("").astype(str).str.strip()

birth["_birth_city_label"] = birth.apply(
    lambda r: (
        f"{r['_birth_city']}, {r['_birth_province']}"
        if r["_birth_city"] and r["_birth_province"]
        else (r["_birth_city"] or r["_birth_province"])
    ),
    axis=1
)

BIRTH_YEAR_MIN = int(birth["_birth_year"].min())
BIRTH_YEAR_MAX = int(birth["_birth_year"].max())

educ = educ_raw[
    educ_raw["NameID"].notna() &
    educ_raw["University_clean"].notna()
].copy()

educ["_student"] = educ["NameID"].astype(str).str.strip()
educ["_uni"] = educ["University_clean"].astype(str).str.strip()
educ["_year"] = pd.to_numeric(educ["Year"], errors="coerce")
educ["_country_class"] = educ["Country"].apply(normalize_country)
educ["_level_cat"] = educ["Level"].apply(normalize_level)
educ["_discipline"] = educ["discipline"].fillna("").astype(str).str.strip()

print("Birth-year range:", BIRTH_YEAR_MIN, "to", BIRTH_YEAR_MAX)
print("Known birth years:", birth["_birth_year"].notna().sum())
print("Unknown birth years:", birth["_birth_year"].isna().sum())


Birth-year range: 1876 to 1927
Known birth years: 488
Unknown birth years: 1125


## 2. Broad Field taxonomy for educational records

In [3]:

FIELD_MAP = {
    # Humanities
    "Drama": "Humanities",
    "English": "Humanities",
    "Fine Arts": "Humanities",
    "History": "Humanities",
    "Language": "Humanities",
    "Literature": "Humanities",
    "Music": "Humanities",
    "Philosophy": "Humanities",
    "Photography": "Humanities",
    "Theology": "Humanities",

    # Social Sciences
    "Anthropology": "Social Sciences",
    "Child Welfare": "Social Sciences",
    "Economics": "Social Sciences",
    "Foreign Service": "Social Sciences",
    "Geography": "Social Sciences",
    "Journalism": "Social Sciences",
    "Law": "Social Sciences",
    "Political Science": "Social Sciences",
    "Psychology": "Social Sciences",
    "Social Science": "Social Sciences",
    "Social Welfare": "Social Sciences",
    "Sociology": "Social Sciences",

    # Business & Administration
    "Accounting": "Business & Administration",
    "Banking & Finance": "Business & Administration",
    "Business": "Business & Administration",
    "Industrial Management": "Business & Administration",
    "Public Administration": "Business & Administration",
    "Railroad Administration": "Business & Administration",
    "Transportation": "Business & Administration",

    # Engineering
    "Aeronautical Engineering": "Engineering",
    "Automotive Engineering": "Engineering",
    "Chemical Engineering": "Engineering",
    "Civil Engineering": "Engineering",
    "Electrical Engineering": "Engineering",
    "Engineering": "Engineering",
    "Hydroelectric Engineering": "Engineering",
    "Marine Enginerring": "Engineering",
    "Mechanical Engineering": "Engineering",
    "Metallurgy": "Engineering",
    "Mining Engineering": "Engineering",
    "Radio Engineering": "Engineering",
    "Textile Engineering": "Engineering",

    # Physical Sciences
    "Chemistry": "Physical Sciences",
    "Mathematics": "Physical Sciences",

    # Biological Sciences
    "Bacteriology": "Biological Sciences",
    "Biochemistry": "Biological Sciences",
    "Biology": "Biological Sciences",
    "Botany": "Biological Sciences",
    "Entomology": "Biological Sciences",
    "Parasitology": "Biological Sciences",
    "Physiology": "Biological Sciences",
    "Plant Pathology": "Biological Sciences",
    "Zoology": "Biological Sciences",

    # Earth & Environmental Sciences
    "Climateology": "Earth & Environmental Sciences",
    "Forestry": "Earth & Environmental Sciences",
    "Geology": "Earth & Environmental Sciences",
    "Meteorology": "Earth & Environmental Sciences",
    "Soil Science": "Earth & Environmental Sciences",

    # Agricultural Sciences
    "Agricultural Economics": "Agricultural Sciences",
    "Agriculture": "Agricultural Sciences",
    "Agrononmy": "Agricultural Sciences",
    "Animal Husbandry": "Agricultural Sciences",
    "Horticulture": "Agricultural Sciences",

    # Health Sciences
    "Dentistry": "Health Sciences",
    "Dietetics": "Health Sciences",
    "Hygiene": "Health Sciences",
    "Medicine": "Health Sciences",
    "Neurology": "Health Sciences",
    "Nursing": "Health Sciences",
    "Nutrition": "Health Sciences",
    "Obstetrics": "Health Sciences",
    "Pharmacy": "Health Sciences",
    "Psychiatry": "Health Sciences",
    "Public Health": "Health Sciences",
    "Surgery": "Health Sciences",
    "Veterinary Medicine": "Health Sciences",

    # Architecture & Planning
    "Architecture": "Architecture & Planning",
    "City Planning": "Architecture & Planning",

    # Education
    "Education": "Education",
    "Library Science": "Education",
    "Physical Education": "Education",

    "Home Economics": "Other / Interdisciplinary",
}

def discipline_to_field(discipline):
    d = clean_string(discipline)
    if not d:
        return "Unspecified"
    return FIELD_MAP.get(d, "Other / Interdisciplinary")

educ["_field"] = educ["discipline"].apply(discipline_to_field)

FIELD_CATEGORIES = sorted(
    educ["_field"].unique().tolist(),
    key=lambda x: (x == "Unspecified", x)
)

LEVEL_CATEGORIES = sorted(
    educ["_level_cat"].unique().tolist(),
    key=lambda x: (x == "Unspecified", x)
)

display(educ["_field"].value_counts().rename("records").to_frame())


,records
_field,
Unspecified,1356
Engineering,599
Social Sciences,392
Humanities,137
Biological Sciences,125
Physical Sciences,116
Business & Administration,114
Health Sciences,105
Education,98


## 3. Historical province boundaries without GeoPandas

In [4]:


# Chinese names for historical provinces used in map popups.
PROVINCE_CHINESE = {
    "Anhui": "安徽",
    "Chahar": "察哈爾",
    "Chahaer": "察哈爾",
    "Fujian": "福建",
    "Gansu": "甘肅",
    "Guangdong": "廣東",
    "Guangxi": "廣西",
    "Guizhou": "貴州",
    "Hebei": "河北",
    "Heilongjiang": "黑龍江",
    "Henan": "河南",
    "Hubei": "湖北",
    "Hunan": "湖南",
    "Jiangsu": "江蘇",
    "Jiangxi": "江西",
    "Jilin": "吉林",
    "Liaoning": "遼寧",
    "Rehe": "熱河",
    "Jehol": "熱河",
    "Shaanxi": "陜西",
    "Shandong": "山東",
    "Shanxi": "山西",
    "Sichuan": "四川",
    "Suiyuan": "綏遠",
    "Yunnan": "雲南",
    "Zhejiang": "浙江",
    "Ningxia": "寧夏",
    "Qinghai": "青海",
    "Xinjiang": "新疆",
    "Xizang": "西藏",
    "Mongolia": "蒙古",
    "Altai": "阿爾泰",
    "Chuanbian": "川邊",
}

# Explicit aliases between directory province spellings and the historical shapefile.
PROVINCE_NAME_ALIASES = {
    "Chahar": "Chahaer",
    "Rehe": "Jehol",
}
REVERSE_PROVINCE_ALIASES = {
    v: k for k, v in PROVINCE_NAME_ALIASES.items()
}

# Assemble a coherent basename in OUTPUT_DIR because an uploaded duplicate .shp
# may have been renamed while sidecars retained the original basename.
source_shp = (
    HISTORICAL_SHP_ALT_PATH
    if HISTORICAL_SHP_ALT_PATH.exists()
    else HISTORICAL_SHP_PATH
)

working_base = OUTPUT_DIR / "Provinces_1912-1931_complete"
working_shp = working_base.with_suffix(".shp")

for ext in [".shp", ".shx", ".dbf", ".prj", ".CPG", ".sbn", ".sbx"]:
    src = source_shp if ext == ".shp" else HISTORICAL_BASE.with_suffix(ext)
    dst = working_base.with_suffix(ext)

    if src.exists():
        shutil.copy2(src, dst)

required = [
    working_base.with_suffix(".shp"),
    working_base.with_suffix(".shx"),
    working_base.with_suffix(".dbf"),
    working_base.with_suffix(".prj"),
]

if not all(p.exists() for p in required):
    missing = [p.name for p in required if not p.exists()]
    raise FileNotFoundError(
        "Historical shapefile components missing: " + ", ".join(missing)
    )

# Read CRS and build transformer to WGS84.
prj_wkt = working_base.with_suffix(".prj").read_text(
    encoding="utf-8",
    errors="ignore"
)

source_crs = CRS.from_wkt(prj_wkt)
to_wgs84 = Transformer.from_crs(
    source_crs,
    CRS.from_epsg(4326),
    always_xy=True
)

reader = shapefile.Reader(str(working_shp))
field_names = [
    f[0]
    for f in reader.fields
    if f[0] != "DeletionFlag"
]

print("Shapefile fields:", field_names)
print("Source CRS:", source_crs)

def transform_coords(obj):
    # Recursively transform nested GeoJSON coordinate arrays.
    if (
        isinstance(obj, (list, tuple))
        and len(obj) >= 2
        and isinstance(obj[0], (int, float))
        and isinstance(obj[1], (int, float))
    ):
        x, y = to_wgs84.transform(obj[0], obj[1])
        return [x, y] + list(obj[2:])

    return [
        transform_coords(x)
        for x in obj
    ]

historical_features = []
province_polygon_points = {}

for sr in reader.iterShapeRecords():
    attrs = dict(zip(field_names, list(sr.record)))
    province_hist = clean_string(attrs.get("Province", ""))
    birth_name = REVERSE_PROVINCE_ALIASES.get(
        province_hist,
        province_hist
    )

    geom = sr.shape.__geo_interface__
    transformed_geom = {
        "type": geom["type"],
        "coordinates": transform_coords(
            geom["coordinates"]
        )
    }

    attrs["birth_province_name"] = birth_name
    attrs["province_zh"] = PROVINCE_CHINESE.get(
        birth_name,
        PROVINCE_CHINESE.get(province_hist, "")
    )

    historical_features.append({
        "type": "Feature",
        "properties": attrs,
        "geometry": transformed_geom,
    })

    # Approximate representative point from transformed shape bbox center.
    xmin, ymin, xmax, ymax = sr.shape.bbox
    cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
    lon, lat = to_wgs84.transform(cx, cy)
    province_polygon_points[birth_name] = (lat, lon)

historical_provinces_geojson = {
    "type": "FeatureCollection",
    "features": historical_features,
}

birth_province_values = sorted(
    birth.loc[
        birth["_birth_province"].ne(""),
        "_birth_province"
    ].unique()
)

matched = [
    p for p in birth_province_values
    if p in province_polygon_points
]

print(
    "Birth provinces matched to historical polygons:",
    len(matched), "/", len(birth_province_values)
)
print(
    "Unmatched:",
    [p for p in birth_province_values if p not in province_polygon_points]
)

historical_geojson_path = (
    OUTPUT_DIR / "Provinces_1912-1931.geojson"
)
historical_geojson_path.write_text(
    json.dumps(
        historical_provinces_geojson,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("Saved:", historical_geojson_path)


Shapefile fields: ['Province', 'Pays']
Source CRS: PROJCS["WGS_1984_UTM_Zone_51N",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["false_easting",500000.0],PARAMETER["false_northing",0.0],PARAMETER["central_meridian",123.0],PARAMETER["scale_factor",0.9996],PARAMETER["latitude_of_origin",0.0],UNIT["Meter",1.0]]
Birth provinces matched to historical polygons: 23 / 23
Unmatched: []
Saved: /mnt/data/birth_education_outputs_enhanced/Provinces_1912-1931.geojson


## 4. Supplied city coordinates

In [5]:

def latin_part(value):
    if pd.isna(value):
        return ""

    text = str(value).strip()
    match = re.search(r"[\u3400-\u9fff]", text)

    if match:
        text = text[:match.start()].strip()

    return re.sub(r"\s+", " ", text).strip()

city_coordinates = pd.read_csv(
    CITY_COORD_PATH,
    sep=";"
)

city_coordinates["_province_py"] = (
    city_coordinates["Province"].apply(latin_part)
)

city_coordinates["_city_py"] = (
    city_coordinates["City"].apply(latin_part)
)

# Normalize decimal commas in a small number of coordinates.
city_coordinates["Latitude"] = pd.to_numeric(
    city_coordinates["Latitude"]
    .astype(str)
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

city_coordinates["Longitude"] = pd.to_numeric(
    city_coordinates["Longitude"]
    .astype(str)
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

city_coordinates = city_coordinates.dropna(
    subset=["Latitude", "Longitude"]
).copy()

city_coord_lookup_pair = {
    (
        clean_string(r["_province_py"]).lower(),
        clean_string(r["_city_py"]).lower(),
    ): (
        float(r["Latitude"]),
        float(r["Longitude"]),
    )
    for _, r in city_coordinates.iterrows()
}

city_coord_lookup = {}

for _, r in birth.loc[
    birth["_birth_city"].ne("")
].iterrows():

    key = (
        r["_birth_province"].lower(),
        r["_birth_city"].lower(),
    )

    if key in city_coord_lookup_pair:
        city_coord_lookup[
            r["_birth_city_label"]
        ] = city_coord_lookup_pair[key]

birth_city_pairs = (
    birth.loc[
        birth["_birth_city"].ne(""),
        ["_birth_province", "_birth_city"]
    ]
    .drop_duplicates()
)

matched_pairs = sum(
    (
        r["_birth_province"].lower(),
        r["_birth_city"].lower(),
    ) in city_coord_lookup_pair
    for _, r in birth_city_pairs.iterrows()
)

print(
    "Unique birthplace province-town pairs matched:",
    matched_pairs,
    "/",
    len(birth_city_pairs)
)


def chinese_part(value):
    if pd.isna(value):
        return ""

    text = str(value).strip()
    chars = re.findall(r"[\u3400-\u9fff]+", text)
    return "".join(chars)

# Chinese province names from the supplied city-coordinate file.
province_chinese_map = dict(PROVINCE_CHINESE)

for raw in city_coordinates["Province"].dropna().astype(str).unique():
    py = latin_part(raw)
    zh = chinese_part(raw)

    if py and zh:
        province_chinese_map[py] = zh

# Historical provinces not represented by birthplace cities in the coordinate file.
province_chinese_map.update({
    "Chahar": "察哈爾",
    "Suiyuan": "綏遠",
    "Rehe": "熱河",
    "Chahaer": "察哈爾",
    "Jehol": "熱河",
    "Gansu": "甘肅",
    "Ningxia": "寧夏",
    "Qinghai": "青海",
    "Xinjiang": "新疆",
    "Xizang": "西藏",
    "Mongolia": "蒙古",
    "Altai": "阿爾泰",
    "Chuanbian": "川邊",
})

# City Chinese-name lookup keyed by romanized province/city.
city_chinese_lookup_pair = {
    (
        clean_string(r["_province_py"]).lower(),
        clean_string(r["_city_py"]).lower(),
    ): chinese_part(r["City"])
    for _, r in city_coordinates.iterrows()
}


Unique birthplace province-town pairs matched: 234 / 234


## 5. University display names and coordinates

In [6]:

university_display = {}

for u, g in educ.groupby("_uni"):
    values = [
        clean_string(x)
        for x in g["University"].dropna().tolist()
        if clean_string(x)
    ]

    university_display[u] = (
        Counter(values).most_common(1)[0][0]
        if values else u
    )

university_geo = {}

for u, g in educ.groupby("_uni"):
    gg = g.dropna(subset=["Latitude", "Longitude"]).copy()

    if gg.empty:
        continue

    lat = pd.to_numeric(
        gg["Latitude"], errors="coerce"
    ).median()

    lon = pd.to_numeric(
        gg["Longitude"], errors="coerce"
    ).median()

    if pd.isna(lat) or pd.isna(lon):
        continue

    def modal(col):
        vals = [
            clean_string(x)
            for x in gg[col].dropna().tolist()
            if clean_string(x)
        ]
        return (
            Counter(vals).most_common(1)[0][0]
            if vals else ""
        )

    country = modal("Country")

    university_geo[u] = {
        "university": u,
        "university_label": university_display[u],
        "lat": float(lat),
        "lon": float(lon),
        "country": country,
        "country_class": normalize_country(country),
        "city": modal("City"),
        "state": modal("Province_State"),
    }

print("Geolocated universities:", len(university_geo))


# Reusable place metadata for aggregating education stages at three levels:
# University/School, City/Town, and State/Province.
university_place_meta = {}

for u, g in educ.groupby("_uni"):
    def modal_all(col):
        vals = [
            clean_string(x)
            for x in g[col].dropna().tolist()
            if clean_string(x)
        ]
        return (
            Counter(vals).most_common(1)[0][0]
            if vals else ""
        )

    country = modal_all("Country")

    university_place_meta[u] = {
        "university": u,
        "university_label": university_display.get(u, u),
        "city": modal_all("City"),
        "state": modal_all("Province_State"),
        "country": country,
        "country_class": normalize_country(country),
    }

# Median coordinates of all geolocated universities in each city/state.
_city_points = defaultdict(list)
_state_points = defaultdict(list)

for u, meta in university_geo.items():
    country = clean_string(meta["country"])
    state = clean_string(meta["state"])
    city = clean_string(meta["city"])

    if city:
        city_key = "|||".join([country, state, city])
        _city_points[city_key].append(
            (meta["lat"], meta["lon"])
        )

    if state:
        state_key = "|||".join([country, state])
        _state_points[state_key].append(
            (meta["lat"], meta["lon"])
        )

EDUCATION_CITY_CENTROIDS = {
    key: {
        "lat": float(np.median([p[0] for p in pts])),
        "lon": float(np.median([p[1] for p in pts])),
    }
    for key, pts in _city_points.items()
}

EDUCATION_STATE_CENTROIDS = {
    key: {
        "lat": float(np.median([p[0] for p in pts])),
        "lon": float(np.median([p[1] for p in pts])),
    }
    for key, pts in _state_points.items()
}

print(
    "Education place centroids:",
    len(EDUCATION_CITY_CENTROIDS), "cities/towns;",
    len(EDUCATION_STATE_CENTROIDS), "states/provinces"
)


Geolocated universities: 413


Education place centroids: 247 cities/towns; 63 states/provinces


## 6. Birthplace lookup and first-education destinations

In [7]:

birth_student = (
    birth[
        [
            "_student",
            "_birth_year",
            "_birth_province",
            "_birth_city",
            "_birth_city_label",
            "FullName_PY",
        ]
    ]
    .drop_duplicates(subset=["_student"])
    .copy()
)

education_records = (
    educ.dropna(subset=["_year"])
    [["_student", "_uni", "_year", "_country_class"]]
    .drop_duplicates()
    .copy()
)

education_records["_year"] = (
    education_records["_year"].astype(int)
)

def first_destinations_by_scope(records, scope):
    rows = []

    for sid, g in records.groupby("_student"):
        if scope != "overall":
            g = g[g["_country_class"] == scope]

        if g.empty:
            continue

        first_year = int(g["_year"].min())

        for _, r in g[g["_year"] == first_year].iterrows():
            rows.append({
                "student": sid,
                "destination_scope": scope,
                "first_year": first_year,
                "university": r["_uni"],
                "country_class": r["_country_class"],
            })

    return pd.DataFrame(rows)

first_tables = {
    scope: first_destinations_by_scope(
        education_records, scope
    )
    for scope in ["overall", "China", "US", "Other"]
}

for scope, table in first_tables.items():
    print(
        scope,
        "| students:",
        table["student"].nunique() if len(table) else 0,
        "| rows:",
        len(table)
    )


overall | students: 1494 | rows: 1514
China | students: 785 | rows: 785
US | students: 1362 | rows: 1407
Other | students: 107 | rows: 107


In [8]:

first_flow_rows = []

for scope, table in first_tables.items():
    merged = table.merge(
        birth_student,
        left_on="student",
        right_on="_student",
        how="inner",
    )

    for _, r in merged.iterrows():
        u = r["university"]

        if u not in university_geo:
            continue

        dest = university_geo[u]
        province = r["_birth_province"]

        # Province-level origin point from historical polygon bbox center.
        if province in province_polygon_points:
            plat, plon = province_polygon_points[province]

            first_flow_rows.append({
                "student": r["student"],
                "student_name": clean_string(r["FullName_PY"]),
                "birth_year": (
                    None
                    if pd.isna(r["_birth_year"])
                    else int(r["_birth_year"])
                ),
                "origin_level": "Province",
                "origin": province,
                "origin_province": province,
                "origin_lat": plat,
                "origin_lon": plon,
                "origin_coord_quality": "historical province polygon",
                "destination_scope": scope,
                "first_year": int(r["first_year"]),
                **dest,
            })

        # City-level origin from supplied coordinates.
        city_label = r["_birth_city_label"]

        if r["_birth_city"] and city_label in city_coord_lookup:
            clat, clon = city_coord_lookup[city_label]

            first_flow_rows.append({
                "student": r["student"],
                "student_name": clean_string(r["FullName_PY"]),
                "birth_year": (
                    None
                    if pd.isna(r["_birth_year"])
                    else int(r["_birth_year"])
                ),
                "origin_level": "City",
                "origin": city_label,
                "origin_province": province,
                "origin_lat": clat,
                "origin_lon": clon,
                "origin_coord_quality": "supplied city coordinate",
                "destination_scope": scope,
                "first_year": int(r["first_year"]),
                **dest,
            })

first_flows = pd.DataFrame(first_flow_rows)

print("First-flow records:", len(first_flows))
print("Students:", first_flows["student"].nunique())
display(first_flows.head())


First-flow records: 5925
Students: 1432


,student,student_name,birth_year,origin_level,origin,origin_province,origin_lat,origin_lon,origin_coord_quality,destination_scope,first_year,university,university_label,lat,lon,country,country_class,city,state
0,N00001,Ding Kangjinmei,1918.0,Province,Jiangsu,Jiangsu,32.929057,119.245465,historical province polygon,overall,1943,University of Michigan,University of Michigan,42.27590,-83.73100,United States,US,Ann Arbor,Michigan
1,N00001,Ding Kangjinmei,1918.0,City,"Chuansha, Jiangsu",Jiangsu,31.199900,121.697200,supplied city coordinate,overall,1943,University of Michigan,University of Michigan,42.27590,-83.73100,United States,US,Ann Arbor,Michigan
2,N00002,Ding Wenhua,NaN,Province,Jiangsu,Jiangsu,32.929057,119.245465,historical province polygon,overall,1943,Mount Holyoke College,Mount Holyoke College,42.25842,-72.57453,United States,US,South Hadley,Massachusetts
3,N00002,Ding Wenhua,NaN,City,"Wujin, Jiangsu",Jiangsu,31.778100,119.964300,supplied city coordinate,overall,1943,Mount Holyoke College,Mount Holyoke College,42.25842,-72.57453,United States,US,South Hadley,Massachusetts
4,N00003,Ding Chen,1919.0,Province,Jiangsu,Jiangsu,32.929057,119.245465,historical province polygon,overall,1939,Jiaotong University,交通大學,31.23040,121.47370,China,China,Shanghai,Shanghai


## 7. Multi-stage trajectories

In [9]:

trajectory_rows = []

for sid, g in education_records.groupby("_student"):
    years = sorted(g["_year"].unique().tolist())

    if not years:
        continue

    selected_years = years[:3]

    stage_lists = [
        g[g["_year"] == y]["_uni"]
        .drop_duplicates()
        .tolist()
        for y in selected_years
    ]

    b = birth_student[
        birth_student["_student"] == sid
    ]

    if b.empty:
        continue

    b = b.iloc[0]

    for combo in itertools.product(*stage_lists):
        row = {
            "student": sid,
            "student_name": clean_string(b["FullName_PY"]),
            "birth_year": (
                None
                if pd.isna(b["_birth_year"])
                else int(b["_birth_year"])
            ),
            "birth_province": b["_birth_province"],
            "birth_city": b["_birth_city"],
            "birth_city_label": b["_birth_city_label"],
        }

        for stage, (year, u) in enumerate(
            zip(selected_years, combo),
            start=1,
        ):
            row[f"stage{stage}_year"] = int(year)
            row[f"stage{stage}_university"] = u
            row[f"stage{stage}_label"] = (
                university_display.get(u, u)
            )
            place_meta = university_place_meta.get(u, {})

            row[f"stage{stage}_country"] = (
                place_meta.get("country_class", "Unknown")
            )
            row[f"stage{stage}_country_name"] = (
                place_meta.get("country", "")
            )
            row[f"stage{stage}_city"] = (
                place_meta.get("city", "")
            )
            row[f"stage{stage}_state"] = (
                place_meta.get("state", "")
            )

        trajectory_rows.append(row)

trajectories = pd.DataFrame(trajectory_rows)

for stage in [1, 2, 3]:
    for suffix, default in [
        ("year", np.nan),
        ("university", ""),
        ("label", ""),
        ("country", ""),
        ("country_name", ""),
        ("city", ""),
        ("state", ""),
    ]:
        col = f"stage{stage}_{suffix}"

        if col not in trajectories.columns:
            trajectories[col] = default

print("Trajectory rows:", len(trajectories))
print("Students:", trajectories["student"].nunique())
display(trajectories.head())


Trajectory rows: 1559
Students: 1494


,student,student_name,birth_year,birth_province,birth_city,birth_city_label,stage1_year,stage1_university,stage1_label,stage1_country,stage1_country_name,stage1_city,stage1_state,stage2_year,stage2_university,stage2_label,stage2_country,stage2_country_name,stage2_city,stage2_state,stage3_year,stage3_university,stage3_label,stage3_country,stage3_country_name,stage3_city,stage3_state
0,N00001,Ding Kangjinmei,1918.0,Jiangsu,Chuansha,"Chuansha, Jiangsu",1943,University of Michigan,University of Michigan,US,United States,Ann Arbor,Michigan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,N00002,Ding Wenhua,NaN,Jiangsu,Wujin,"Wujin, Jiangsu",1943,Mount Holyoke College,Mount Holyoke College,US,United States,South Hadley,Massachusetts,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,N00003,Ding Chen,1919.0,Jiangsu,Wuxi,"Wuxi, Jiangsu",1939,Jiaotong University,交通大學,China,China,Shanghai,Shanghai,1943.0,Harvard University,Harvard University,US,United States,Cambridge,Massachusetts,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,N00004,Ding Meiceng,NaN,Jiangsu,,Jiangsu,1943,University of California,University of California,US,United States,Berkeley,California,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,N00005,Ding Enmu,NaN,Jiangsu,Shanghai,"Shanghai, Jiangsu",1940,St. John's University,聖約翰大學,China,China,Shanghai,Jiangsu,1942.0,Stanford University,Stanford University,US,United States,Stanford,California,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. HTML helper functions

In [10]:

def json_compact(obj):
    return json.dumps(
        obj,
        ensure_ascii=False,
        separators=(",", ":")
    )

def js_birth_year_filter():
    # Shared JS fragment used by the interactive visualizations.
    return f"""
function birthYearOK(d){{
    let y1=parseInt(birthYearMin.value,10);
    let y2=parseInt(birthYearMax.value,10);

    if(y1>y2) [y1,y2]=[y2,y1];

    if(d.birth_year===null || d.birth_year===undefined || d.birth_year===""){{
        return unknownBirthYear.value==="include";
    }}

    const y=+d.birth_year;
    return y>=y1 && y<=y2;
}}
"""


## 9. Sankey: birthplace → first education

In [11]:

first_sankey_records = first_flows[
    [
        "student",
        "birth_year",
        "origin_level",
        "origin",
        "destination_scope",
        "university",
        "university_label",
        "country_class",
        "country",
        "city",
        "state",
        "first_year",
    ]
].to_dict("records")

FIRST_SANKEY_HTML = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Birthplace to first education</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body{{margin:0;font-family:Inter,system-ui,Arial,sans-serif;color:#17202a}}
header{{padding:14px 18px;border-bottom:1px solid #ddd}}
h1{{font-size:19px;margin:0 0 5px}}.sub{{font-size:12px;color:#667085}}
.controls{{display:flex;flex-wrap:wrap;gap:10px;padding:10px 14px;border-bottom:1px solid #ddd;align-items:end}}
.control{{display:flex;flex-direction:column;gap:3px}}
label{{font-size:11px;font-weight:650;color:#475467}}
select,input,button{{font:inherit;padding:7px 8px;border:1px solid #cfd4dc;border-radius:6px;background:white}}
#chart{{height:780px}}.stat{{margin-left:auto;padding:8px;font-size:12px;color:#667085}}
</style>
</head>
<body>
<header>
<h1>Birthplace → First Education</h1>
<div class="sub">Flow width = distinct students. Year filter refers to year of birth.</div>
</header>

<div class="controls">
<div class="control">
<label>Birthplace level</label>
<select id="originLevel">
<option value="Province">Province</option>
<option value="City">City / town</option>
</select>
</div>

<div class="control">
<label>First education</label>
<select id="scope">
<option value="overall">Overall</option>
<option value="China">In China</option>
<option value="US">In the United States</option>
<option value="Other">In other countries</option>
</select>
</div>

<div class="control">
<label>Education stage geography</label>
<select id="educationLevel">
<option value="university">University / school</option>
<option value="city">City / town</option>
<option value="state">State / province</option>
</select>
</div>

<div class="control">
<label>Birth year from</label>
<input id="birthYearMin" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MIN}">
</div>

<div class="control">
<label>Birth year to</label>
<input id="birthYearMax" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MAX}">
</div>

<div class="control">
<label>Unknown birth year</label>
<select id="unknownBirthYear">
<option value="include">Include</option>
<option value="exclude">Exclude</option>
</select>
</div>

<div class="control">
<label>Top N flows</label>
<input id="topN" type="number" min="5" max="300" value="50">
</div>

<div class="control">
<label>Minimum students</label>
<input id="minStudents" type="number" min="1" max="100" value="1">
</div>

<button id="reset">Reset</button>
<div class="stat" id="stat"></div>
</div>

<div id="chart"></div>

<script>
const DATA={json_compact(first_sankey_records)};
{js_birth_year_filter()}


function educationPlace(d,mode){{
    if(mode==="university"){{
        return {{
            key:"U:"+d.university,
            label:d.university_label
        }};
    }}

    if(mode==="city"){{
        const city=(d.city||"").trim();
        if(!city) return null;

        const label=[
            city,
            (d.state||"").trim(),
            (d.country||"").trim()
        ].filter(Boolean).join(", ");

        return {{
            key:"C:"+[
                d.country||"",
                d.state||"",
                city
            ].join("|||"),
            label
        }};
    }}

    const state=(d.state||"").trim();
    if(!state) return null;

    return {{
        key:"S:"+[
            d.country||"",
            state
        ].join("|||"),
        label:[
            state,
            (d.country||"").trim()
        ].filter(Boolean).join(", ")
    }};
}}

function render(){{
    const origin=originLevel.value;
    const sc=scope.value;
    const eduMode=educationLevel.value;
    const top=Math.max(5,parseInt(topN.value||"50",10));
    const minS=Math.max(1,parseInt(minStudents.value||"1",10));

    const rows=DATA.filter(d=>
        birthYearOK(d) &&
        d.origin_level===origin &&
        d.destination_scope===sc
    );

    const pairs=new Map();
    const meta=new Map();

    rows.forEach(d=>{{
        const place=educationPlace(d,eduMode);
        if(!place) return;

        const key=d.origin+"|||"+place.key;

        if(!pairs.has(key)){{
            pairs.set(key,new Set());
            meta.set(key,{{
                origin:d.origin,
                destination_key:place.key,
                destination_label:place.label
            }});
        }}

        pairs.get(key).add(d.student);
    }});

    let flows=[...pairs.entries()].map(([key,set])=>({{
        ...meta.get(key),
        students:set.size
    }}));

    flows=flows
        .filter(d=>d.students>=minS)
        .sort((a,b)=>b.students-a.students)
        .slice(0,top);

    const nodeKeys=[];
    const nodeLabels={{}};

    flows.forEach(d=>{{
        const o="O:"+d.origin;
        const u="E:"+d.destination_key;

        if(!nodeKeys.includes(o)) nodeKeys.push(o);
        if(!nodeKeys.includes(u)) nodeKeys.push(u);

        nodeLabels[o]=d.origin;
        nodeLabels[u]=d.destination_label;
    }});

    const idx=Object.fromEntries(nodeKeys.map((x,i)=>[x,i]));

    const trace={{
        type:"sankey",
        arrangement:"snap",
        node:{{
            pad:14,
            thickness:16,
            label:nodeKeys.map(k=>nodeLabels[k]),
            line:{{color:"rgba(40,40,40,.35)",width:.5}}
        }},
        link:{{
            source:flows.map(d=>idx["O:"+d.origin]),
            target:flows.map(d=>idx["E:"+d.destination_key]),
            value:flows.map(d=>d.students),
            customdata:flows.map(
                d=>`${{d.origin}} → ${{d.destination_label}}: ${{d.students}} students`
            ),
            hovertemplate:"%{{customdata}}<extra></extra>"
        }}
    }};

    Plotly.react(
        "chart",
        [trace],
        {{margin:{{l:25,r:25,t:20,b:25}},font:{{size:10}},autosize:true}},
        {{responsive:true,displaylogo:false}}
    );

    stat.textContent=`${{flows.length}} flows · ${{rows.length}} qualifying records`;
}}

["originLevel","scope","educationLevel","birthYearMin","birthYearMax","unknownBirthYear","topN","minStudents"]
.forEach(id=>document.getElementById(id).addEventListener("change",render));

reset.onclick=()=>{{
    originLevel.value="Province";
    scope.value="overall";
    educationLevel.value="university";
    birthYearMin.value="{BIRTH_YEAR_MIN}";
    birthYearMax.value="{BIRTH_YEAR_MAX}";
    unknownBirthYear.value="include";
    topN.value=50;
    minStudents.value=1;
    render();
}};

render();
</script>
</body>
</html>
"""

first_sankey_path = OUTPUT_DIR / "01_birth_to_first_education_sankey.html"
first_sankey_path.write_text(FIRST_SANKEY_HTML, encoding="utf-8")
print("Saved:", first_sankey_path)


Saved: /mnt/data/birth_education_outputs_enhanced/01_birth_to_first_education_sankey.html


## 10. Spatial map: birthplace → first education

In [12]:

first_map_records = first_flows[
    [
        "student",
        "birth_year",
        "origin_level",
        "origin",
        "origin_lat",
        "origin_lon",
        "origin_coord_quality",
        "destination_scope",
        "university",
        "university_label",
        "lat",
        "lon",
        "country_class",
        "country",
        "city",
        "state",
        "first_year",
    ]
].replace({np.nan: None}).to_dict("records")

FIRST_MAP_HTML = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Birthplace to first education map</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
body{{margin:0;font-family:Inter,system-ui,Arial,sans-serif;color:#17202a}}
header{{padding:14px 18px;border-bottom:1px solid #ddd}}
h1{{font-size:19px;margin:0 0 5px}}.sub{{font-size:12px;color:#667085}}
.controls{{display:flex;flex-wrap:wrap;gap:10px;padding:10px 14px;border-bottom:1px solid #ddd;align-items:end}}
.control{{display:flex;flex-direction:column;gap:3px}}
label{{font-size:11px;font-weight:650;color:#475467}}
select,input,button{{font:inherit;padding:7px 8px;border:1px solid #cfd4dc;border-radius:6px;background:white}}
#map{{height:740px}}.stat{{margin-left:auto;padding:8px;font-size:12px;color:#667085}}
</style>
</head>
<body>
<header>
<h1>Birthplace → First Education Spatial Flows</h1>
<div class="sub">Year filter refers to year of birth. Historical 1912–1931 province boundaries are shown.</div>
</header>

<div class="controls">
<div class="control">
<label>Birthplace level</label>
<select id="originLevel">
<option value="Province">Province</option>
<option value="City">City / town</option>
</select>
</div>

<div class="control">
<label>First education</label>
<select id="scope">
<option value="overall">Overall</option>
<option value="China">In China</option>
<option value="US">In the United States</option>
<option value="Other">In other countries</option>
</select>
</div>

<div class="control">
<label>Education stage geography</label>
<select id="educationLevel">
<option value="university">University / school</option>
<option value="city">City / town</option>
<option value="state">State / province</option>
</select>
</div>

<div class="control">
<label>Birth year from</label>
<input id="birthYearMin" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MIN}">
</div>

<div class="control">
<label>Birth year to</label>
<input id="birthYearMax" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MAX}">
</div>

<div class="control">
<label>Unknown birth year</label>
<select id="unknownBirthYear">
<option value="include">Include</option>
<option value="exclude">Exclude</option>
</select>
</div>

<div class="control">
<label>Top N flows</label>
<input id="topN" type="number" min="5" max="300" value="75">
</div>

<div class="control">
<label>Minimum students</label>
<input id="minStudents" type="number" min="1" max="100" value="1">
</div>

<button id="reset">Reset</button>
<div class="stat" id="stat"></div>
</div>

<div id="map"></div>

<script>
const DATA={json_compact(first_map_records)};
const HISTORICAL_PROVINCES={json_compact(historical_provinces_geojson)};
const CITY_CENTROIDS={json_compact(EDUCATION_CITY_CENTROIDS)};
const STATE_CENTROIDS={json_compact(EDUCATION_STATE_CENTROIDS)};
{js_birth_year_filter()}

const map=L.map("map",{{worldCopyJump:true}}).setView([30,25],2);

L.tileLayer(
    "https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png",
    {{maxZoom:18,attribution:"© OpenStreetMap contributors"}}
).addTo(map);

L.geoJSON(
    HISTORICAL_PROVINCES,
    {{
        style:()=>({{
            color:"#596273",
            weight:1,
            opacity:.7,
            fillOpacity:.025
        }}),
        onEachFeature:(feature,layer)=>{{
            const p=feature.properties||{{}};
            layer.bindTooltip(
                p.birth_province_name || p.Province || ""
            );
        }}
    }}
).addTo(map);

let layer=L.layerGroup().addTo(map);


function educationPoint(d,mode){{
    if(mode==="university"){{
        return {{
            key:"U:"+d.university,
            label:d.university_label,
            lat:d.lat,
            lon:d.lon
        }};
    }}

    if(mode==="city"){{
        const city=(d.city||"").trim();
        if(!city) return null;

        const rawKey=[
            d.country||"",
            d.state||"",
            city
        ].join("|||");

        const coord=CITY_CENTROIDS[rawKey];
        if(!coord) return null;

        return {{
            key:"C:"+rawKey,
            label:[
                city,
                (d.state||"").trim(),
                (d.country||"").trim()
            ].filter(Boolean).join(", "),
            lat:coord.lat,
            lon:coord.lon
        }};
    }}

    const state=(d.state||"").trim();
    if(!state) return null;

    const rawKey=[
        d.country||"",
        state
    ].join("|||");

    const coord=STATE_CENTROIDS[rawKey];
    if(!coord) return null;

    return {{
        key:"S:"+rawKey,
        label:[
            state,
            (d.country||"").trim()
        ].filter(Boolean).join(", "),
        lat:coord.lat,
        lon:coord.lon
    }};
}}

function render(){{
    layer.clearLayers();

    const origin=originLevel.value;
    const sc=scope.value;
    const eduMode=educationLevel.value;
    const top=Math.max(5,parseInt(topN.value||"75",10));
    const minS=Math.max(1,parseInt(minStudents.value||"1",10));

    const rows=DATA.filter(d=>
        birthYearOK(d) &&
        d.origin_level===origin &&
        d.destination_scope===sc
    );

    const pairs=new Map();
    const meta=new Map();

    rows.forEach(d=>{{
        const place=educationPoint(d,eduMode);
        if(!place) return;

        const key=d.origin+"|||"+place.key;

        if(!pairs.has(key)){{
            pairs.set(key,new Set());
            meta.set(key,{{
                ...d,
                destination_key:place.key,
                destination_label:place.label,
                destination_lat:place.lat,
                destination_lon:place.lon
            }});
        }}

        pairs.get(key).add(d.student);
    }});

    let flows=[...pairs.entries()].map(([key,set])=>(
        {{...meta.get(key),students:set.size}}
    ));

    flows=flows
        .filter(d=>d.students>=minS)
        .sort((a,b)=>b.students-a.students)
        .slice(0,top);

    const bounds=[];

    flows.forEach(d=>{{
        L.polyline(
            [[d.origin_lat,d.origin_lon],[d.destination_lat,d.destination_lon]],
            {{
                weight:Math.max(.7,Math.min(8,.7+Math.sqrt(d.students)*1.2)),
                opacity:.35
            }}
        )
        .bindTooltip(`${{d.origin}} → ${{d.destination_label}}: ${{d.students}} students`)
        .addTo(layer);

        L.circleMarker(
            [d.origin_lat,d.origin_lon],
            {{radius:5,weight:1,fillOpacity:.75}}
        )
        .bindTooltip(`<b>${{d.origin}}</b><br>${{d.origin_coord_quality}}`)
        .addTo(layer);

        L.circleMarker(
            [d.destination_lat,d.destination_lon],
            {{radius:5,weight:1,fillOpacity:.75}}
        )
        .bindTooltip(`<b>${{d.destination_label}}</b>`)
        .addTo(layer);

        bounds.push([d.origin_lat,d.origin_lon],[d.destination_lat,d.destination_lon]);
    }});

    if(bounds.length){{
        map.fitBounds(bounds,{{padding:[30,30]}});
    }}

    stat.textContent=`${{flows.length}} displayed flows · ${{rows.length}} qualifying records`;
}}

["originLevel","scope","educationLevel","birthYearMin","birthYearMax","unknownBirthYear","topN","minStudents"]
.forEach(id=>document.getElementById(id).addEventListener("change",render));

reset.onclick=()=>{{
    originLevel.value="Province";
    scope.value="overall";
    educationLevel.value="university";
    birthYearMin.value="{BIRTH_YEAR_MIN}";
    birthYearMax.value="{BIRTH_YEAR_MAX}";
    unknownBirthYear.value="include";
    topN.value=75;
    minStudents.value=1;
    render();
}};

render();
</script>
</body>
</html>
"""

first_map_path = OUTPUT_DIR / "02_birth_to_first_education_map.html"
first_map_path.write_text(FIRST_MAP_HTML, encoding="utf-8")
print("Saved:", first_map_path)


Saved: /mnt/data/birth_education_outputs_enhanced/02_birth_to_first_education_map.html


## 11. Multi-stage Sankey

In [13]:

multi_records = (
    trajectories
    .replace({np.nan: ""})
    .to_dict("records")
)

MULTI_SANKEY_HTML = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Birthplace multi-stage trajectories</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body{{margin:0;font-family:Inter,system-ui,Arial,sans-serif;color:#17202a}}
header{{padding:14px 18px;border-bottom:1px solid #ddd}}
h1{{font-size:19px;margin:0 0 5px}}.sub{{font-size:12px;color:#667085}}
.controls{{display:flex;flex-wrap:wrap;gap:10px;padding:10px 14px;border-bottom:1px solid #ddd;align-items:end}}
.control{{display:flex;flex-direction:column;gap:3px}}
label{{font-size:11px;font-weight:650;color:#475467}}
select,input,button{{font:inherit;padding:7px 8px;border:1px solid #cfd4dc;border-radius:6px;background:white}}
#chart{{height:850px}}.stat{{margin-left:auto;padding:8px;font-size:12px;color:#667085}}
</style>
</head>
<body>

<header>
<h1>Birthplace → Stage 1 → Stage 2 → Stage 3</h1>
<div class="sub">Stages are the first three distinct observed education years. Year filter refers to year of birth.</div>
</header>

<div class="controls">
<div class="control">
<label>Birthplace level</label>
<select id="originLevel">
<option value="Province">Province</option>
<option value="City">City / town</option>
</select>
</div>

<div class="control">
<label>Education stage geography</label>
<select id="educationLevel">
<option value="university">University / school</option>
<option value="city">City / town</option>
<option value="state">State / province</option>
</select>
</div>

<div class="control">
<label>Birth year from</label>
<input id="birthYearMin" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MIN}">
</div>

<div class="control">
<label>Birth year to</label>
<input id="birthYearMax" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MAX}">
</div>

<div class="control">
<label>Unknown birth year</label>
<select id="unknownBirthYear">
<option value="include">Include</option>
<option value="exclude">Exclude</option>
</select>
</div>

<div class="control">
<label>Top N links per stage</label>
<input id="topN" type="number" min="5" max="300" value="50">
</div>

<div class="control">
<label>Minimum students</label>
<input id="minStudents" type="number" min="1" max="100" value="1">
</div>

<button id="reset">Reset</button>
<div class="stat" id="stat"></div>
</div>

<div id="chart"></div>

<script>
const DATA={json_compact(multi_records)};
{js_birth_year_filter()}

function originFor(row,level){{
    return level==="Province"
        ? (row.birth_province||"")
        : (row.birth_city_label||"");
}}


function stagePlace(r,stage,mode){{
    const u=r[`stage${{stage}}_university`]||"";
    if(!u) return null;

    if(mode==="university"){{
        return {{
            key:u,
            label:r[`stage${{stage}}_label`]||u
        }};
    }}

    if(mode==="city"){{
        const city=(r[`stage${{stage}}_city`]||"").trim();
        if(!city) return null;

        const state=(r[`stage${{stage}}_state`]||"").trim();
        const country=(r[`stage${{stage}}_country_name`]||"").trim();

        return {{
            key:[country,state,city].join("|||"),
            label:[city,state,country].filter(Boolean).join(", ")
        }};
    }}

    const state=(r[`stage${{stage}}_state`]||"").trim();
    if(!state) return null;

    const country=(r[`stage${{stage}}_country_name`]||"").trim();

    return {{
        key:[country,state].join("|||"),
        label:[state,country].filter(Boolean).join(", ")
    }};
}}

function aggregate(rows,sourceFn,targetFn){{
    const m=new Map();

    rows.forEach(r=>{{
        const s=sourceFn(r);
        const t=targetFn(r);

        if(!s || !t) return;

        const key=s+"|||"+t;

        if(!m.has(key)) m.set(key,new Set());
        m.get(key).add(r.student);
    }});

    return [...m.entries()].map(([key,set])=>{{
        const [source,target]=key.split("|||");
        return {{source,target,students:set.size}};
    }});
}}

function render(){{
    const level=originLevel.value;
    const eduMode=educationLevel.value;
    const top=Math.max(5,parseInt(topN.value||"50",10));
    const minS=Math.max(1,parseInt(minStudents.value||"1",10));

    const rows=DATA.filter(
        r=>birthYearOK(r)
    );

    let l0=aggregate(
        rows,
        r=>originFor(r,level) ? "O:"+originFor(r,level) : "",
        r=>{{
            const p=stagePlace(r,1,eduMode);
            return p ? "S1:"+p.key : "";
        }}
    );

    let l1=aggregate(
        rows,
        r=>{{
            const p=stagePlace(r,1,eduMode);
            return p ? "S1:"+p.key : "";
        }},
        r=>{{
            const p=stagePlace(r,2,eduMode);
            return p ? "S2:"+p.key : "";
        }}
    );

    let l2=aggregate(
        rows,
        r=>{{
            const p=stagePlace(r,2,eduMode);
            return p ? "S2:"+p.key : "";
        }},
        r=>{{
            const p=stagePlace(r,3,eduMode);
            return p ? "S3:"+p.key : "";
        }}
    );

    function selectTop(arr){{
        return arr
            .filter(d=>d.students>=minS)
            .sort((a,b)=>b.students-a.students)
            .slice(0,top);
    }}

    l0=selectTop(l0);
    l1=selectTop(l1);
    l2=selectTop(l2);

    const links=[...l0,...l1,...l2];
    const nodeKeys=[];
    const nodeLabels={{}};
    const uniLabel={{}};

    DATA.forEach(r=>{{
        [1,2,3].forEach(stage=>{{
            const p=stagePlace(r,stage,eduMode);
            if(p) uniLabel[`S${{stage}}:`+p.key]=p.label;
        }});
    }});

    links.forEach(d=>{{
        if(!nodeKeys.includes(d.source)) nodeKeys.push(d.source);
        if(!nodeKeys.includes(d.target)) nodeKeys.push(d.target);

        nodeLabels[d.source]=d.source.startsWith("O:")
            ? d.source.slice(2)
            : (uniLabel[d.source]||d.source);

        nodeLabels[d.target]=d.target.startsWith("O:")
            ? d.target.slice(2)
            : (uniLabel[d.target]||d.target);
    }});

    const idx=Object.fromEntries(nodeKeys.map((x,i)=>[x,i]));

    const trace={{
        type:"sankey",
        arrangement:"fixed",
        node:{{
            pad:14,
            thickness:16,
            label:nodeKeys.map(k=>nodeLabels[k]),
            x:nodeKeys.map(k=>
                k.startsWith("O:") ? 0.01 :
                k.startsWith("S1:") ? 0.34 :
                k.startsWith("S2:") ? 0.67 : 0.99
            ),
            y:nodeKeys.map((k,i)=>(
                (i+1)/(nodeKeys.length+1)
            )),
            line:{{color:"rgba(40,40,40,.35)",width:.5}}
        }},
        link:{{
            source:links.map(d=>idx[d.source]),
            target:links.map(d=>idx[d.target]),
            value:links.map(d=>d.students),
            customdata:links.map(
                d=>`${{nodeLabels[d.source]}} → ${{nodeLabels[d.target]}}: ${{d.students}} students`
            ),
            hovertemplate:"%{{customdata}}<extra></extra>"
        }}
    }};

    Plotly.react(
        "chart",
        [trace],
        {{margin:{{l:25,r:25,t:20,b:25}},font:{{size:10}},autosize:true}},
        {{responsive:true,displaylogo:false}}
    );

    stat.textContent=`${{links.length}} displayed links · ${{rows.length}} qualifying trajectory records · missing birthplace trajectories begin at Stage 1`;
}}

["originLevel","educationLevel","birthYearMin","birthYearMax","unknownBirthYear","topN","minStudents"]
.forEach(id=>document.getElementById(id).addEventListener("change",render));

reset.onclick=()=>{{
    originLevel.value="Province";
    educationLevel.value="university";
    birthYearMin.value="{BIRTH_YEAR_MIN}";
    birthYearMax.value="{BIRTH_YEAR_MAX}";
    unknownBirthYear.value="include";
    topN.value=50;
    minStudents.value=1;
    render();
}};

render();
</script>
</body>
</html>
"""

multi_sankey_path = OUTPUT_DIR / "03_multistage_sankey.html"
multi_sankey_path.write_text(MULTI_SANKEY_HTML, encoding="utf-8")
print("Saved:", multi_sankey_path)


Saved: /mnt/data/birth_education_outputs_enhanced/03_multistage_sankey.html


## 12. Multi-stage spatial map with stage selection and stage colors

In [14]:

multistage_map_rows = []

for _, r in trajectories.iterrows():
    for origin_level in ["Province", "City"]:

        if origin_level == "Province":
            origin = r["birth_province"]

            if origin not in province_polygon_points:
                continue

            olat, olon = province_polygon_points[origin]
            quality = "historical province polygon"

        else:
            if not r["birth_city"]:
                continue

            origin = r["birth_city_label"]

            if origin not in city_coord_lookup:
                continue

            olat, olon = city_coord_lookup[origin]
            quality = "supplied city coordinate"

        row = {
            "student": r["student"],
            "student_name": r["student_name"],
            "birth_year": r["birth_year"],
            "origin_level": origin_level,
            "origin": origin,
            "origin_lat": olat,
            "origin_lon": olon,
            "origin_coord_quality": quality,
        }

        for stage in [1,2,3]:
            u = r.get(f"stage{stage}_university", "")

            if u and u in university_geo:
                row[f"stage{stage}_university"] = u
                row[f"stage{stage}_label"] = university_display[u]
                row[f"stage{stage}_lat"] = university_geo[u]["lat"]
                row[f"stage{stage}_lon"] = university_geo[u]["lon"]
                row[f"stage{stage}_country"] = university_geo[u]["country_class"]
                row[f"stage{stage}_country_name"] = university_place_meta.get(u, {}).get("country", "")
                row[f"stage{stage}_city"] = university_place_meta.get(u, {}).get("city", "")
                row[f"stage{stage}_state"] = university_place_meta.get(u, {}).get("state", "")
            else:
                row[f"stage{stage}_university"] = ""
                row[f"stage{stage}_label"] = ""
                row[f"stage{stage}_lat"] = np.nan
                row[f"stage{stage}_lon"] = np.nan
                row[f"stage{stage}_country"] = ""
                row[f"stage{stage}_country_name"] = ""
                row[f"stage{stage}_city"] = ""
                row[f"stage{stage}_state"] = ""

        multistage_map_rows.append(row)

multistage_map_df = pd.DataFrame(multistage_map_rows)

print("Map-ready trajectory rows:", len(multistage_map_df))


Map-ready trajectory rows: 2406


In [15]:

multi_map_records = (
    multistage_map_df
    .replace({np.nan: None})
    .to_dict("records")
)

MULTI_MAP_HTML = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Multi-stage spatial trajectories</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
body{{margin:0;font-family:Inter,system-ui,Arial,sans-serif;color:#17202a}}
header{{padding:14px 18px;border-bottom:1px solid #ddd}}
h1{{font-size:19px;margin:0 0 5px}}.sub{{font-size:12px;color:#667085}}
.controls{{display:flex;flex-wrap:wrap;gap:10px;padding:10px 14px;border-bottom:1px solid #ddd;align-items:end}}
.control{{display:flex;flex-direction:column;gap:3px}}
label{{font-size:11px;font-weight:650;color:#475467}}
select,input,button{{font:inherit;padding:7px 8px;border:1px solid #cfd4dc;border-radius:6px;background:white}}
select[multiple]{{height:70px;min-width:165px}}
#map{{height:770px}}.stat{{margin-left:auto;padding:8px;font-size:12px;color:#667085}}
.legend{{background:white;padding:7px 9px;border:1px solid #ddd;border-radius:6px;line-height:1.5}}
</style>
</head>
<body>
<header>
<h1>Birthplace → Multi-stage Education Trajectories</h1>
<div class="sub">Select one or more trajectory stages. Optional stage colors distinguish the displayed segments.</div>
</header>

<div class="controls">
<div class="control">
<label>Birthplace level</label>
<select id="originLevel">
<option value="Province">Province</option>
<option value="City">City / town</option>
</select>
</div>

<div class="control">
<label>Education stage geography</label>
<select id="educationLevel">
<option value="university">University / school</option>
<option value="city">City / town</option>
<option value="state">State / province</option>
</select>
</div>

<div class="control">
<label>Stages</label>
<select id="stageFilter" multiple>
<option value="0" selected>Birth → Stage 1</option>
<option value="1" selected>Stage 1 → Stage 2</option>
<option value="2" selected>Stage 2 → Stage 3</option>
</select>
</div>

<div class="control">
<label>Stage colors</label>
<select id="stageColors">
<option value="on">Different colors</option>
<option value="off">Single color</option>
</select>
</div>

<div class="control">
<label>Birth year from</label>
<input id="birthYearMin" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MIN}">
</div>

<div class="control">
<label>Birth year to</label>
<input id="birthYearMax" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MAX}">
</div>

<div class="control">
<label>Unknown birth year</label>
<select id="unknownBirthYear">
<option value="include">Include</option>
<option value="exclude">Exclude</option>
</select>
</div>

<div class="control">
<label>Minimum students</label>
<input id="minStudents" type="number" min="1" max="100" value="2">
</div>

<div class="control">
<label>Top N per stage</label>
<input id="topN" type="number" min="5" max="300" value="60">
</div>

<button id="reset">Reset</button>
<div class="stat" id="stat"></div>
</div>

<div id="map"></div>

<script>
const DATA={json_compact(multi_map_records)};
const HISTORICAL_PROVINCES={json_compact(historical_provinces_geojson)};
const CITY_CENTROIDS={json_compact(EDUCATION_CITY_CENTROIDS)};
const STATE_CENTROIDS={json_compact(EDUCATION_STATE_CENTROIDS)};
{js_birth_year_filter()}

const STAGE_COLORS={{
    "0":"#4c78a8",
    "1":"#f28e2b",
    "2":"#59a14f"
}};

const STAGE_LABELS={{
    "0":"Birth → Stage 1",
    "1":"Stage 1 → Stage 2",
    "2":"Stage 2 → Stage 3"
}};

const map=L.map("map",{{worldCopyJump:true}}).setView([30,25],2);

L.tileLayer(
    "https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png",
    {{maxZoom:18,attribution:"© OpenStreetMap contributors"}}
).addTo(map);

L.geoJSON(
    HISTORICAL_PROVINCES,
    {{
        style:()=>({{
            color:"#596273",
            weight:1,
            opacity:.65,
            fillOpacity:.02
        }})
    }}
).addTo(map);

let layer=L.layerGroup().addTo(map);
let legend=null;

function selectedStages(){{
    return [...stageFilter.selectedOptions].map(o=>o.value);
}}


function stagePoint(r,stage,mode){{
    const u=r[`stage${{stage}}_university`]||"";
    if(!u) return null;

    if(mode==="university"){{
        return {{
            key:`U:${{u}}`,
            lat:r[`stage${{stage}}_lat`],
            lon:r[`stage${{stage}}_lon`],
            label:r[`stage${{stage}}_label`]||u
        }};
    }}

    if(mode==="city"){{
        const city=(r[`stage${{stage}}_city`]||"").trim();
        if(!city) return null;

        const state=(r[`stage${{stage}}_state`]||"").trim();
        const country=(r[`stage${{stage}}_country_name`]||"").trim();
        const rawKey=[country,state,city].join("|||");
        const coord=CITY_CENTROIDS[rawKey];

        if(!coord) return null;

        return {{
            key:`C:${{rawKey}}`,
            lat:coord.lat,
            lon:coord.lon,
            label:[city,state,country].filter(Boolean).join(", ")
        }};
    }}

    const state=(r[`stage${{stage}}_state`]||"").trim();
    if(!state) return null;

    const country=(r[`stage${{stage}}_country_name`]||"").trim();
    const rawKey=[country,state].join("|||");
    const coord=STATE_CENTROIDS[rawKey];

    if(!coord) return null;

    return {{
        key:`S:${{rawKey}}`,
        lat:coord.lat,
        lon:coord.lon,
        label:[state,country].filter(Boolean).join(", ")
    }};
}}

function aggregate(rows,sourceFn,targetFn,labelFn,stage){{
    const m=new Map();
    const meta=new Map();

    rows.forEach(r=>{{
        const s=sourceFn(r);
        const t=targetFn(r);

        if(!s || !t) return;

        const key=s.key+"|||"+t.key;

        if(!m.has(key)){{
            m.set(key,new Set());
            meta.set(key,{{
                source:s,
                target:t,
                label:labelFn(r),
                stage
            }});
        }}

        m.get(key).add(r.student);
    }});

    return [...m.entries()].map(([key,set])=>({{
        ...meta.get(key),
        students:set.size
    }}));
}}

function render(){{
    layer.clearLayers();

    if(legend){{
        map.removeControl(legend);
        legend=null;
    }}

    const level=originLevel.value;
    const eduMode=educationLevel.value;
    const stages=selectedStages();
    const useStageColors=stageColors.value==="on";
    const minS=Math.max(1,parseInt(minStudents.value||"2",10));
    const top=Math.max(5,parseInt(topN.value||"60",10));

    const rows=DATA.filter(
        r=>r.origin_level===level && birthYearOK(r)
    );

    const arrays={{}};

    arrays["0"]=aggregate(
        rows,
        r=>({{
            key:"O:"+r.origin,
            lat:r.origin_lat,
            lon:r.origin_lon,
            label:r.origin
        }}),
        r=>stagePoint(r,1,eduMode),
        r=>{{
            const p=stagePoint(r,1,eduMode);
            return `${{r.origin}} → ${{p ? p.label : ""}}`;
        }},
        "0"
    );

    arrays["1"]=aggregate(
        rows,
        r=>stagePoint(r,1,eduMode),
        r=>stagePoint(r,2,eduMode),
        r=>{{
            const p1=stagePoint(r,1,eduMode);
            const p2=stagePoint(r,2,eduMode);
            return `${{p1 ? p1.label : ""}} → ${{p2 ? p2.label : ""}}`;
        }},
        "1"
    );

    arrays["2"]=aggregate(
        rows,
        r=>stagePoint(r,2,eduMode),
        r=>stagePoint(r,3,eduMode),
        r=>{{
            const p2=stagePoint(r,2,eduMode);
            const p3=stagePoint(r,3,eduMode);
            return `${{p2 ? p2.label : ""}} → ${{p3 ? p3.label : ""}}`;
        }},
        "2"
    );

    const segments=[];

    stages.forEach(stage=>{{
        arrays[stage]
            .filter(d=>d.students>=minS)
            .sort((a,b)=>b.students-a.students)
            .slice(0,top)
            .forEach(d=>segments.push(d));
    }});

    const bounds=[];

    segments.forEach(d=>{{
        if(
            d.source.lat==null || d.source.lon==null ||
            d.target.lat==null || d.target.lon==null
        ) return;

        const color=useStageColors
            ? STAGE_COLORS[d.stage]
            : "#4c78a8";

        L.polyline(
            [[d.source.lat,d.source.lon],[d.target.lat,d.target.lon]],
            {{
                weight:Math.max(.7,Math.min(8,.7+Math.sqrt(d.students)*1.2)),
                opacity:.42,
                color
            }}
        )
        .bindTooltip(
            `${{STAGE_LABELS[d.stage]}}<br>${{d.label}}<br>${{d.students}} students`
        )
        .addTo(layer);

        bounds.push(
            [d.source.lat,d.source.lon],
            [d.target.lat,d.target.lon]
        );
    }});

    if(bounds.length){{
        map.fitBounds(bounds,{{padding:[30,30]}});
    }}

    if(useStageColors && stages.length){{
        legend=L.control({{position:"bottomright"}});

        legend.onAdd=function(){{
            const div=L.DomUtil.create("div","legend");
            div.innerHTML="<b>Stages</b><br>"+
                stages.map(s=>
                    `<span style="display:inline-block;width:10px;height:10px;background:${{STAGE_COLORS[s]}};margin-right:5px"></span>${{STAGE_LABELS[s]}}`
                ).join("<br>");
            return div;
        }};

        legend.addTo(map);
    }}

    stat.textContent=
        `${{segments.length}} displayed segments · `+
        `${{rows.length}} qualifying trajectory records`;
}}

["originLevel","educationLevel","stageFilter","stageColors","birthYearMin","birthYearMax","unknownBirthYear","minStudents","topN"]
.forEach(id=>document.getElementById(id).addEventListener("change",render));

reset.onclick=()=>{{
    originLevel.value="Province";
    educationLevel.value="university";
    [...stageFilter.options].forEach(o=>o.selected=true);
    stageColors.value="on";
    birthYearMin.value="{BIRTH_YEAR_MIN}";
    birthYearMax.value="{BIRTH_YEAR_MAX}";
    unknownBirthYear.value="include";
    minStudents.value=2;
    topN.value=60;
    render();
}};

render();
</script>
</body>
</html>
"""

multi_map_path = OUTPUT_DIR / "04_multistage_spatial_map.html"
multi_map_path.write_text(MULTI_MAP_HTML, encoding="utf-8")
print("Saved:", multi_map_path)


Saved: /mnt/data/birth_education_outputs_enhanced/04_multistage_spatial_map.html


## 13. Combined map: distribution of births by province and city/town

In [16]:

# Combined data for province choropleth + proportional city circles.

birth_province_records = (
    birth.loc[
        birth["_birth_province"].ne(""),
        ["_student", "_birth_year", "_birth_province"]
    ]
    .rename(columns={
        "_student":"student",
        "_birth_year":"birth_year",
        "_birth_province":"province"
    })
    .copy()
)

birth_province_records["birth_year"] = (
    birth_province_records["birth_year"]
    .apply(lambda x: None if pd.isna(x) else int(x))
)

birth_province_records["province_zh"] = (
    birth_province_records["province"]
    .map(province_chinese_map)
    .fillna("")
)

birth_city_rows = []

for _, r in birth.loc[
    birth["_birth_city"].ne("")
].iterrows():

    label = r["_birth_city_label"]

    if label not in city_coord_lookup:
        continue

    lat, lon = city_coord_lookup[label]
    key = (
        r["_birth_province"].lower(),
        r["_birth_city"].lower(),
    )

    birth_city_rows.append({
        "student": r["_student"],
        "birth_year": (
            None
            if pd.isna(r["_birth_year"])
            else int(r["_birth_year"])
        ),
        "city": label,
        "city_py": r["_birth_city"],
        "city_zh": city_chinese_lookup_pair.get(key, ""),
        "province": r["_birth_province"],
        "province_zh": province_chinese_map.get(
            r["_birth_province"], ""
        ),
        "lat": lat,
        "lon": lon,
    })

birth_city_records = pd.DataFrame(birth_city_rows)

birth_province_payload = birth_province_records.to_dict("records")
birth_city_payload = birth_city_records.to_dict("records")

BIRTH_COMBINED_MAP_HTML = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Birth distribution in China</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
body{{margin:0;font-family:Inter,system-ui,Arial,sans-serif;color:#17202a}}
header{{padding:14px 18px;border-bottom:1px solid #ddd}}
h1{{font-size:19px;margin:0 0 5px}}.sub{{font-size:12px;color:#667085}}
.controls{{display:flex;flex-wrap:wrap;gap:10px;padding:10px 14px;border-bottom:1px solid #ddd;align-items:end}}
.control{{display:flex;flex-direction:column;gap:3px}}
label{{font-size:11px;font-weight:650;color:#475467}}
select,input,button{{font:inherit;padding:7px 8px;border:1px solid #cfd4dc;border-radius:6px;background:white}}
#map{{height:740px}}.stat{{margin-left:auto;padding:8px;font-size:12px;color:#667085}}
</style>
</head>
<body>

<header>
<h1>Distribution of Birthplaces in China</h1>
<div class="sub">
Historical provinces use a blue choropleth; city/town circles use a contrasting orange palette. Switch between provinces, cities, or both. Popups show romanized and Chinese place names.
</div>
</header>

<div class="controls">

<div class="control">
<label>Display</label>
<select id="displayMode">
<option value="both">Provinces + cities</option>
<option value="provinces">Provinces only</option>
<option value="cities">Cities only</option>
</select>
</div>

<div class="control">
<label>Birth year from</label>
<input id="birthYearMin" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MIN}">
</div>

<div class="control">
<label>Birth year to</label>
<input id="birthYearMax" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MAX}">
</div>

<div class="control">
<label>Unknown birth year</label>
<select id="unknownBirthYear">
<option value="include">Include</option>
<option value="exclude">Exclude</option>
</select>
</div>

<button id="exportProvinceTable">Export province table</button>
<button id="exportCityTable">Export city table</button>
<button id="reset">Reset</button>

<div class="stat" id="stat"></div>
</div>

<div id="map"></div>

<script>
const PROVINCE_DATA={json_compact(birth_province_payload)};
const CITY_DATA={json_compact(birth_city_payload)};
const HISTORICAL_PROVINCES={json_compact(historical_provinces_geojson)};
{js_birth_year_filter()}

const map=L.map("map").setView([35,105],4);

L.tileLayer(
    "https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png",
    {{maxZoom:18,attribution:"© OpenStreetMap contributors"}}
).addTo(map);

let provinceLayer=null;
let cityLayer=L.layerGroup().addTo(map);
let currentProvinceTable=[];
let currentCityTable=[];

function colorFor(v,maxV){{
    if(maxV<=0) return "#f2f4f7";
    const t=Math.max(0,Math.min(1,v/maxV));
    const light=96-t*54;
    return `hsl(212,60%,${{light}}%)`;
}}

function computeProvinceTable(){{
    const counts=new Map();

    PROVINCE_DATA
        .filter(birthYearOK)
        .forEach(d=>{{
            if(!counts.has(d.province)){{
                counts.set(d.province,{{
                    province:d.province,
                    province_zh:d.province_zh||"",
                    students:0
                }});
            }}

            counts.get(d.province).students += 1;
        }});

    return [...counts.values()]
        .sort((a,b)=>b.students-a.students);
}}

function computeCityTable(){{
    const counts=new Map();

    CITY_DATA
        .filter(birthYearOK)
        .forEach(d=>{{
            if(!counts.has(d.city)){{
                counts.set(d.city,{{
                    city:d.city,
                    city_py:d.city_py,
                    city_zh:d.city_zh||"",
                    province:d.province,
                    province_zh:d.province_zh||"",
                    lat:d.lat,
                    lon:d.lon,
                    students:0
                }});
            }}

            counts.get(d.city).students += 1;
        }});

    return [...counts.values()]
        .sort((a,b)=>b.students-a.students);
}}

function render(){{
    const mode=displayMode.value;

    currentProvinceTable=computeProvinceTable();
    currentCityTable=computeCityTable();

    const pMap=Object.fromEntries(
        currentProvinceTable.map(d=>[d.province,d])
    );

    const maxV=Math.max(
        1,
        ...currentProvinceTable.map(d=>d.students)
    );

    if(provinceLayer){{
        map.removeLayer(provinceLayer);
        provinceLayer=null;
    }}

    cityLayer.clearLayers();

    if(mode==="provinces" || mode==="both"){{
        provinceLayer=L.geoJSON(
            HISTORICAL_PROVINCES,
            {{
                style:feature=>{{
                    const name=feature.properties.birth_province_name;
                    const v=(pMap[name]?.students)||0;

                    return {{
                        color:"#596273",
                        weight:1,
                        fillColor:colorFor(v,maxV),
                        fillOpacity:.78
                    }};
                }},
                onEachFeature:(feature,layer)=>{{
                    const p=feature.properties||{{}};
                    const name=p.birth_province_name||p.Province||"";
                    const zh=p.province_zh||"";
                    const v=(pMap[name]?.students)||0;

                    layer.bindTooltip(
                        `<b>${{name}}${{zh ? " "+zh : ""}}</b><br>Students born: ${{v}}`,
                        {{sticky:true}}
                    );
                }}
            }}
        ).addTo(map);
    }}

    if(mode==="cities" || mode==="both"){{
        currentCityTable.forEach(d=>{{
            L.circleMarker(
                [d.lat,d.lon],
                {{
                    radius:Math.max(3,2.5+Math.sqrt(d.students)*1.8),
                    color:"#9a4d00",
                    fillColor:"#f28e2b",
                    weight:1.2,
                    fillOpacity:.78
                }}
            )
            .bindTooltip(
                `<b>${{d.city_py}}${{d.city_zh ? " "+d.city_zh : ""}}</b>`+
                `<br>${{d.province}}${{d.province_zh ? " "+d.province_zh : ""}}`+
                `<br>Students born: ${{d.students}}`,
                {{sticky:true}}
            )
            .addTo(cityLayer);
        }});
    }}

    const totalProvince=currentProvinceTable.reduce(
        (a,b)=>a+b.students,0
    );

    const totalCity=currentCityTable.reduce(
        (a,b)=>a+b.students,0
    );

    stat.textContent=
        `${{totalProvince}} province-coded births · `+
        `${{totalCity}} city-coded births · `+
        `${{currentCityTable.length}} cities/towns`;
}}

function downloadCSV(filename,header,rows){{
    const lines=[header.join(",")];

    rows.forEach(row=>{{
        lines.push(
            header.map(h=>{{
                const v=row[h] ?? "";
                return `"${{String(v).replaceAll('"','""')}}"`;
            }}).join(",")
        );
    }});

    const blob=new Blob(
        [lines.join("\\n")],
        {{type:"text/csv;charset=utf-8"}}
    );

    const url=URL.createObjectURL(blob);
    const a=document.createElement("a");
    a.href=url;
    a.download=filename;
    a.click();
    URL.revokeObjectURL(url);
}}

exportProvinceTable.onclick=()=>downloadCSV(
    "births_by_historical_province_filtered.csv",
    ["province","province_zh","students"],
    currentProvinceTable
);

exportCityTable.onclick=()=>downloadCSV(
    "births_by_city_filtered.csv",
    [
        "city_py","city_zh",
        "province","province_zh",
        "students","lat","lon"
    ],
    currentCityTable
);

["displayMode","birthYearMin","birthYearMax","unknownBirthYear"]
.forEach(id=>document.getElementById(id).addEventListener("change",render));

reset.onclick=()=>{{
    displayMode.value="both";
    birthYearMin.value="{BIRTH_YEAR_MIN}";
    birthYearMax.value="{BIRTH_YEAR_MAX}";
    unknownBirthYear.value="include";
    render();
}};

render();
</script>
</body>
</html>
"""

birth_combined_map_path = (
    OUTPUT_DIR / "05_birth_distribution_china_combined.html"
)

birth_combined_map_path.write_text(
    BIRTH_COMBINED_MAP_HTML,
    encoding="utf-8"
)

print("Saved:", birth_combined_map_path)


Saved: /mnt/data/birth_education_outputs_enhanced/05_birth_distribution_china_combined.html


## 14. Combined birth map continuation

In [17]:

# The former standalone city map has been merged with the historical-province
# choropleth into:
#   05_birth_distribution_china_combined.html
#
# birth_city_records is created in the combined-map cell above and is reused
# below for statistical exports.
print("City and province birth maps are combined in:", birth_combined_map_path)


City and province birth maps are combined in: /mnt/data/birth_education_outputs_enhanced/05_birth_distribution_china_combined.html


## 15. U.S. educational places by state

In [18]:

US_STATE_ABBR = {
    "Alabama":"AL","Alaska":"AK","Arizona":"AZ","Arkansas":"AR","California":"CA",
    "Colorado":"CO","Connecticut":"CT","Delaware":"DE","District of Columbia":"DC",
    "Florida":"FL","Georgia":"GA","Hawaii":"HI","Idaho":"ID","Illinois":"IL",
    "Indiana":"IN","Iowa":"IA","Kansas":"KS","Kentucky":"KY","Louisiana":"LA",
    "Maine":"ME","Maryland":"MD","Massachusetts":"MA","Michigan":"MI","Minnesota":"MN",
    "Mississippi":"MS","Missouri":"MO","Montana":"MT","Nebraska":"NE","Nevada":"NV",
    "New Hampshire":"NH","New Jersey":"NJ","New Mexico":"NM","New York":"NY",
    "North Carolina":"NC","North Dakota":"ND","Ohio":"OH","Oklahoma":"OK","Oregon":"OR",
    "Pennsylvania":"PA","Rhode Island":"RI","South Carolina":"SC","South Dakota":"SD",
    "Tennessee":"TN","Texas":"TX","Utah":"UT","Vermont":"VT","Virginia":"VA",
    "Washington":"WA","West Virginia":"WV","Wisconsin":"WI","Wyoming":"WY"
}

birth_year_lookup = (
    birth_student
    .set_index("_student")["_birth_year"]
    .to_dict()
)

us_educ = educ[
    educ["_country_class"] == "US"
].copy()

us_educ["birth_year"] = us_educ["_student"].map(
    birth_year_lookup
)

us_educ["state"] = (
    us_educ["Province_State"]
    .fillna("")
    .astype(str)
    .str.strip()
)

us_educ["state_abbr"] = us_educ["state"].map(
    US_STATE_ABBR
)

us_educ = us_educ[
    us_educ["state_abbr"].notna()
].copy()

us_educ["university_label"] = us_educ["_uni"].map(
    university_display
)

us_state_payload = []

for _, r in us_educ.iterrows():
    us_state_payload.append({
        "student": r["_student"],
        "birth_year": (
            None
            if pd.isna(r["birth_year"])
            else int(r["birth_year"])
        ),
        "state": r["state"],
        "state_abbr": r["state_abbr"],
        "university": r["_uni"],
        "university_label": r["university_label"],
        "field": r["_field"],
        "level": r["_level_cat"],
    })

print("U.S. education records with mapped state:", len(us_state_payload))
print("States represented:", us_educ["state_abbr"].nunique())


U.S. education records with mapped state:

 2008
States represented: 42


In [19]:

def option_tags(values):
    return "".join(
        f'<option value="{html.escape(str(v), quote=True)}">{html.escape(str(v))}</option>'
        for v in values
    )

US_STATE_MAP_HTML = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>U.S. educational places by state</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body{{margin:0;font-family:Inter,system-ui,Arial,sans-serif;color:#17202a}}
header{{padding:14px 18px;border-bottom:1px solid #ddd}}
h1{{font-size:19px;margin:0 0 5px}}.sub{{font-size:12px;color:#667085}}
.controls{{display:flex;flex-wrap:wrap;gap:10px;padding:10px 14px;border-bottom:1px solid #ddd;align-items:end}}
.control{{display:flex;flex-direction:column;gap:3px}}
label{{font-size:11px;font-weight:650;color:#475467}}
select,input,button{{font:inherit;padding:7px 8px;border:1px solid #cfd4dc;border-radius:6px;background:white}}
select[multiple]{{height:70px;min-width:155px}}
#chart{{height:720px}}.stat{{margin-left:auto;padding:8px;font-size:12px;color:#667085}}
</style>
</head>
<body>
<header>
<h1>Distribution of U.S. Educational Places by State</h1>
<div class="sub">Filter by student birth year, broad Field, and normalized Level. Metric can be students, records, or distinct universities.</div>
</header>

<div class="controls">
<div class="control">
<label>Birth year from</label>
<input id="birthYearMin" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MIN}">
</div>

<div class="control">
<label>Birth year to</label>
<input id="birthYearMax" type="number" min="{BIRTH_YEAR_MIN}" max="{BIRTH_YEAR_MAX}" value="{BIRTH_YEAR_MAX}">
</div>

<div class="control">
<label>Unknown birth year</label>
<select id="unknownBirthYear">
<option value="include">Include</option>
<option value="exclude">Exclude</option>
</select>
</div>

<div class="control">
<label>Field(s)</label>
<select id="fieldFilter" multiple>
{option_tags(FIELD_CATEGORIES)}
</select>
<span style="font-size:11px;color:#667085">No selection = all</span>
</div>

<div class="control">
<label>Level(s)</label>
<select id="levelFilter" multiple>
{option_tags(LEVEL_CATEGORIES)}
</select>
<span style="font-size:11px;color:#667085">No selection = all</span>
</div>

<div class="control">
<label>Map metric</label>
<select id="metric">
<option value="students">Distinct students</option>
<option value="records">Education records</option>
<option value="universities">Distinct universities</option>
</select>
</div>

<button id="exportTable">Export table CSV</button>
<button id="reset">Reset</button>
<div class="stat" id="stat"></div>
</div>

<div id="chart"></div>

<script>
const DATA={json_compact(us_state_payload)};
{js_birth_year_filter()}

function selectedValues(el){{
    return [...el.selectedOptions].map(o=>o.value);
}}

let currentTable=[];

function computeTable(){{
    const fields=selectedValues(fieldFilter);
    const levels=selectedValues(levelFilter);

    const rows=DATA.filter(d=>
        birthYearOK(d) &&
        (fields.length===0 || fields.includes(d.field)) &&
        (levels.length===0 || levels.includes(d.level))
    );

    const stateMap=new Map();

    rows.forEach(d=>{{
        if(!stateMap.has(d.state_abbr)){{
            stateMap.set(d.state_abbr,{{
                state:d.state,
                state_abbr:d.state_abbr,
                students:new Set(),
                universities:new Set(),
                records:0
            }});
        }}

        const x=stateMap.get(d.state_abbr);
        x.students.add(d.student);
        x.universities.add(d.university);
        x.records += 1;
    }});

    return [...stateMap.values()].map(d=>({{
        state:d.state,
        state_abbr:d.state_abbr,
        students:d.students.size,
        universities:d.universities.size,
        records:d.records
    }})).sort((a,b)=>b.students-a.students);
}}

function render(){{
    currentTable=computeTable();
    const m=metric.value;

    const trace={{
        type:"choropleth",
        locationmode:"USA-states",
        locations:currentTable.map(d=>d.state_abbr),
        z:currentTable.map(d=>d[m]),
        text:currentTable.map(
            d=>`${{d.state}}<br>Students: ${{d.students}}<br>Universities: ${{d.universities}}<br>Records: ${{d.records}}`
        ),
        hovertemplate:"%{{text}}<extra></extra>",
        colorbar:{{title:m}}
    }};

    const layout={{
        geo:{{
            scope:"usa",
            projection:{{type:"albers usa"}},
            showlakes:true
        }},
        margin:{{l:10,r:10,t:10,b:10}}
    }};

    Plotly.react(
        "chart",
        [trace],
        layout,
        {{responsive:true,displaylogo:false}}
    );

    stat.textContent=
        `${{currentTable.reduce((a,b)=>a+b.students,0)}} summed state-level student counts · `+
        `${{currentTable.length}} states`;
}}

function exportCSV(){{
    const lines=["state,state_abbr,distinct_students,distinct_universities,education_records"];

    currentTable.forEach(d=>{{
        lines.push(
            `"${{String(d.state).replaceAll('"','""')}}",${{d.state_abbr}},${{d.students}},${{d.universities}},${{d.records}}`
        );
    }});

    const blob=new Blob([lines.join("\\n")],{{type:"text/csv;charset=utf-8"}});
    const url=URL.createObjectURL(blob);
    const a=document.createElement("a");
    a.href=url;
    a.download="us_education_by_state_filtered.csv";
    a.click();
    URL.revokeObjectURL(url);
}}

["birthYearMin","birthYearMax","unknownBirthYear","fieldFilter","levelFilter","metric"]
.forEach(id=>document.getElementById(id).addEventListener("change",render));

exportTable.onclick=exportCSV;

reset.onclick=()=>{{
    birthYearMin.value="{BIRTH_YEAR_MIN}";
    birthYearMax.value="{BIRTH_YEAR_MAX}";
    unknownBirthYear.value="include";
    [...fieldFilter.options].forEach(o=>o.selected=false);
    [...levelFilter.options].forEach(o=>o.selected=false);
    metric.value="students";
    render();
}};

render();
</script>
</body>
</html>
"""

us_state_map_path = OUTPUT_DIR / "07_us_education_by_state.html"
us_state_map_path.write_text(
    US_STATE_MAP_HTML,
    encoding="utf-8"
)

print("Saved:", us_state_map_path)


Saved: /mnt/data/birth_education_outputs_enhanced/07_us_education_by_state.html


## 16. Export statistical tables

In [20]:

# Static full-sample tables are exported here.
# The interactive birth and U.S.-state maps additionally provide an
# "Export table CSV" button that downloads the currently filtered table.

birth_province_table = (
    birth.loc[
        birth["_birth_province"].ne(""),
        ["_student", "_birth_year", "_birth_province"]
    ]
    .groupby("_birth_province")
    .agg(
        students=("_student", "nunique"),
        known_birth_year=("_birth_year", lambda x: x.notna().sum()),
        unknown_birth_year=("_birth_year", lambda x: x.isna().sum()),
    )
    .reset_index()
    .rename(columns={"_birth_province":"province"})
    .sort_values("students", ascending=False)
)

birth_city_table = (
    birth_city_records
    .groupby(["city","province"], as_index=False)
    .agg(
        students=("student","nunique"),
        latitude=("lat","first"),
        longitude=("lon","first"),
    )
    .sort_values("students", ascending=False)
)

us_state_table = (
    us_educ
    .groupby(["state","state_abbr"], as_index=False)
    .agg(
        distinct_students=("_student","nunique"),
        distinct_universities=("_uni","nunique"),
        education_records=("_student","size"),
    )
    .sort_values("distinct_students", ascending=False)
)

birth_province_table.to_csv(
    OUTPUT_DIR / "table_births_by_historical_province.csv",
    index=False
)

birth_city_table.to_csv(
    OUTPUT_DIR / "table_births_by_city.csv",
    index=False
)

us_state_table.to_csv(
    OUTPUT_DIR / "table_us_education_by_state.csv",
    index=False
)

first_flows.to_csv(
    OUTPUT_DIR / "birth_to_first_education_records.csv",
    index=False
)

trajectories.to_csv(
    OUTPUT_DIR / "multistage_trajectory_records.csv",
    index=False
)

multistage_map_df.to_csv(
    OUTPUT_DIR / "multistage_map_records.csv",
    index=False
)

print("Birth province table:")
display(birth_province_table.head(20))

print("Birth city table:")
display(birth_city_table.head(20))

print("U.S. state education table:")
display(us_state_table.head(20))


Birth province table:


,province,students,known_birth_year,unknown_birth_year
3,Guangdong,574,119,455
11,Jiangsu,304,115,189
22,Zhejiang,139,52,87
6,Hebei,126,37,89
2,Fujian,96,30,66
10,Hunan,42,21,21
17,Shandong,40,14,26
9,Hubei,34,12,22
0,Anhui,29,11,18
12,Jiangxi,26,9,17


Birth city table:


,city,province,students,latitude,longitude
166,"Taishan, Guangdong",Guangdong,112,22.2516,112.7786
144,"Shanghai, Jiangsu",Jiangsu,71,31.2304,121.4737
231,"Zhongshan, Guangdong",Guangdong,38,22.5176,113.3926
203,"Xinhui, Guangdong",Guangdong,37,22.4580,113.0340
186,"Wuxi, Jiangsu",Jiangsu,31,31.4906,120.3119
170,"Tianjin, Hebei",Hebei,30,39.3434,117.3616
100,"Kaiping, Guangdong",Guangdong,27,22.3762,112.6991
188,"Wuxian, Jiangsu",Jiangsu,20,31.3191,120.6291
121,"Nanhai, Guangdong",Guangdong,18,23.0277,113.1429
185,"Wujin, Jiangsu",Jiangsu,18,31.7781,119.9643


U.S. state education table:


,state,state_abbr,distinct_students,distinct_universities,education_records
28,New York,NY,302,30,343
18,Massachusetts,MA,282,23,301
3,California,CA,251,22,292
19,Michigan,MI,166,11,185
10,Illinois,IL,142,18,158
11,Indiana,IN,88,7,93
34,Pennsylvania,PA,77,14,84
31,Ohio,OH,54,16,61
41,Wisconsin,WI,51,10,53
20,Minnesota,MN,50,5,59


## 17. Preview interactive outputs

In [21]:
display(IFrame(src=str(first_sankey_path), width="100%", height=850))

In [22]:
display(IFrame(src=str(first_map_path), width="100%", height=850))

In [23]:
display(IFrame(src=str(multi_sankey_path), width="100%", height=850))

In [24]:
display(IFrame(src=str(multi_map_path), width="100%", height=850))

In [25]:
display(IFrame(src=str(birth_combined_map_path), width="100%", height=850))

In [26]:
# City map merged into birth_combined_map_path; no separate preview needed.

In [27]:
display(IFrame(src=str(us_state_map_path), width="100%", height=850))


## Interpretation cautions

- Birth-year filters use the directory's `birth_year` field; most records have no reported birth year, so the include/exclude-unknown control materially affects sample size.
- Education sequence is reconstructed from the education `Year` field and may not capture every institution attended.
- Multiple institutions in the same education year are preserved rather than forced into an arbitrary order.
- City/town birth coordinates come from the supplied historical-place coordinate table.
- Province polygons come from the supplied 1912–1931 shapefile. `Chahar` is matched to `Chahaer`, and `Rehe` to `Jehol`.
- The U.S. state choropleth counts only education records whose state value can be mapped to a standard U.S. state abbreviation.
